In [18]:
import pandas as pd
import numpy as np
import re
import string

In [19]:
df = pd.read_csv("/content/news_zomato.csv")
print(df.head())

                                                text     date  \
0  MC Pro Inside Edge: HNIs try to recall 'Khul j...  unknown   
1  Zomato rolls back green uniform | What led to ...  unknown   
2   Why Zomato is a fundamental investor's nightmare  unknown   
3  Zomato CEO Deepinder Goyal marries Mexican ent...  unknown   
4  Swiggy clears the air on viral post mocking Zo...  unknown   

                source sentiment  url  
0  Moneycontrol_Kaggle       NaN  NaN  
1  Moneycontrol_Kaggle       NaN  NaN  
2  Moneycontrol_Kaggle       NaN  NaN  
3  Moneycontrol_Kaggle       NaN  NaN  
4  Moneycontrol_Kaggle       NaN  NaN  


In [20]:
print(df.shape)

(221, 5)


In [21]:
df.columns

Index(['text', 'date', 'source', 'sentiment', 'url'], dtype='object')

In [22]:
df.isnull().sum()

,0
text,0
date,0
source,0
sentiment,161
url,161


In [23]:
print(f"\nData types:\n{df.dtypes}")


Data types:
text         object
date         object
source       object
sentiment    object
url          object
dtype: object


In [24]:
df['date'] = df['date'].replace('unknown', np.nan)

/tmp/ipykernel_3092/3883815477.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['date'] = df['date'].replace('unknown', np.nan)


In [25]:
df.drop(columns=['url'], inplace=True)

In [26]:
df['sentiment'] = df['sentiment'].fillna('unknown')

In [27]:
print("After handling missing values:")
print(df.isnull().sum(), "\n")

After handling missing values:
text           0
date         221
source         0
sentiment      0
dtype: int64 



In [28]:
def clean_text(text: str) -> str:
    """
    Clean a single news headline / article text:
    - Lowercase
    - Remove URLs
    - Remove HTML tags
    - Remove punctuation
    - Collapse extra whitespace
    """
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation (keep apostrophes for contractions if needed)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Collapse multiple spaces / newlines
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df['text_clean'] = df['text'].apply(clean_text)

print("Sample cleaned text:")
for raw, clean in zip(df['text'].head(3), df['text_clean'].head(3)):
    print(f"  RAW   : {raw[:80]}")
    print(f"  CLEAN : {clean[:80]}")
    print()

Sample cleaned text:
  RAW   : MC Pro Inside Edge: HNIs try to recall 'Khul ja sim sim', algo trading firms in 
  CLEAN : mc pro inside edge hnis try to recall khul ja sim sim algo trading firms in the 

  RAW   : Zomato rolls back green uniform | What led to the 'green fleet' backlash?
  CLEAN : zomato rolls back green uniform what led to the green fleet backlash

  RAW   : Why Zomato is a fundamental investor's nightmare
  CLEAN : why zomato is a fundamental investors nightmare



In [29]:
df['word_count']       = df['text'].apply(lambda x: len(str(x).split()))
df['clean_word_count'] = df['text_clean'].apply(lambda x: len(x.split()))

# Character length
df['char_length'] = df['text_clean'].apply(len)

# Source encoding (Label Encoding)
source_map = {s: i for i, s in enumerate(df['source'].unique())}
df['source_encoded'] = df['source'].map(source_map)
print("Source label encoding:", source_map)

# Sentiment encoding (only for labeled rows)
sentiment_map = {'positive': 1, 'negative': -1, 'neutral': 0, 'unknown': np.nan}
df['sentiment_encoded'] = df['sentiment'].str.lower().map(sentiment_map)

Source label encoding: {'Moneycontrol_Kaggle': 0, 'HuggingFace_IndianFinancialNews': 1, 'ETMarkets_Kaggle': 2}


In [30]:
before = len(df)
df.drop_duplicates(subset='text_clean', inplace=True)
df = df[df['text_clean'].str.strip() != '']
df.reset_index(drop=True, inplace=True)
print(f"\nRemoved {before - len(df)} duplicate / empty rows. Remaining: {len(df)}")


Removed 0 duplicate / empty rows. Remaining: 221


In [31]:
labeled_df   = df[df['sentiment'] != 'unknown'].copy()
unlabeled_df = df[df['sentiment'] == 'unknown'].copy()

print(f"\nLabeled rows   : {len(labeled_df)}")
print(f"Unlabeled rows : {len(unlabeled_df)}")


Labeled rows   : 60
Unlabeled rows : 161


In [32]:
print("\n" + "=" * 50)
print("PREPROCESSED DATASET")
print("=" * 50)
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
print(f"\nMissing:\n{df.isnull().sum()}")
print(f"\nStats on text length:\n{df[['word_count', 'char_length']].describe()}")
print(f"\nSentiment distribution:\n{df['sentiment'].value_counts()}")
print(f"\nSource distribution:\n{df['source'].value_counts()}")


PREPROCESSED DATASET
Shape   : (221, 10)
Columns : ['text', 'date', 'source', 'sentiment', 'text_clean', 'word_count', 'clean_word_count', 'char_length', 'source_encoded', 'sentiment_encoded']

Missing:
text                   0
date                 221
source                 0
sentiment              0
text_clean             0
word_count             0
clean_word_count       0
char_length            0
source_encoded         0
sentiment_encoded    161
dtype: int64

Stats on text length:
        word_count   char_length
count   221.000000    221.000000
mean    251.027149   1480.701357
std     579.203802   3400.064029
min       6.000000     32.000000
25%      12.000000     68.000000
50%      15.000000     81.000000
75%     328.000000   1928.000000
max    4838.000000  29205.000000

Sentiment distribution:
sentiment
unknown     161
Negative     26
Neutral      17
Positive     17
Name: count, dtype: int64

Source distribution:
source
ETMarkets_Kaggle                   124
HuggingFace_IndianFi

In [33]:
df.to_csv("zomato_preprocessed.csv", index=False)
labeled_df.to_csv("zomato_labeled.csv", index=False)
unlabeled_df.to_csv("zomato_unlabeled.csv", index=False)

print("\nFiles saved:")
print("  - zomato_preprocessed.csv  (full cleaned dataset)")
print("  - zomato_labeled.csv       (rows with sentiment labels)")
print("  - zomato_unlabeled.csv     (rows needing annotation)")


Files saved:
  - zomato_preprocessed.csv  (full cleaned dataset)
  - zomato_labeled.csv       (rows with sentiment labels)
  - zomato_unlabeled.csv     (rows needing annotation)
